In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import glob
import json
import os
import pickle
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import make_axes_locatable

from tsfm_lens.utils import (
    apply_custom_style,
    plot_3d_and_univariate,
    plot_3d_and_univariate_with_predictions,
    plot_forecast_and_rmse,
    plot_single_p_prediction,
)

WORK_DIR = os.getenv("WORK", "")
DATA_DIR = os.path.join(WORK_DIR, "data")

In [ ]:
apply_custom_style(config_path="../../config/plotting.yaml")

In [ ]:
WORK_DIR = os.getenv("WORK", "")
DATA_DIR = os.path.join(WORK_DIR, "data")

In [ ]:
plot_save_dir = os.path.join("../figs", "decoder_edit")
os.makedirs(plot_save_dir, exist_ok=True)


In [ ]:
system_name = "Lorenz"
run_rseed = 42
decoder_edit_dir = os.path.join(
    WORK_DIR, "results", "decoder_edit", system_name, f"rseed-{run_rseed}"
)
metrics_dir = os.path.join("../outputs", f"rseed-{run_rseed}")

In [ ]:
def load_p_data(p: float, decoder_edit_dir: str) -> dict:
    """
    Load prediction data from pickle file for a given p value.
    """
    p_file = os.path.join(decoder_edit_dir, f"edit_with_p_({p:.2f}).pkl")
    if not os.path.exists(p_file):
        closest_files = glob.glob(os.path.join(decoder_edit_dir, "edit_with_p_*.pkl"))
        p_file = min(
            closest_files, key=lambda f: abs(float(f.split("(")[1].split(")")[0]) - p)
        )
    with open(p_file, "rb") as f:
        return pickle.load(f)

In [ ]:
def load_metrics(p: float, metrics_dir: str, metrics_fname: str) -> dict:
    """
    Load metrics from a JSON file.
    """
    metrics_fname = f"{metrics_fname}_p-{p}"
    metrics_path = os.path.join(metrics_dir, f"{metrics_fname}.json")
    print(f"Loading metrics from {metrics_path}")
    if not os.path.exists(metrics_path):
        print(f"Metrics file {metrics_path} does not exist")
        return {}
    with open(metrics_path, "r") as f:
        return json.load(f)

In [ ]:
# Get all p value files and load predictions
all_p_files = glob.glob(os.path.join(decoder_edit_dir, "edit_with_p_*.pkl"))
p_values = sorted([float(f.split("(")[1].split(")")[0]) for f in all_p_files])
p_values = p_values[:11]
print(p_values)

In [ ]:
predictions_decoder_edit_dict = {}
metrics_dict = {}
context_window_idx = 0
chosen_pred_length = 512

ground_truth = None
for i, p in enumerate(p_values):
    data = load_p_data(p, decoder_edit_dir)

    context = data["context"][context_window_idx]
    future_vals = data["future_vals"][context_window_idx]
    # only get the ground truth once
    if i == 0:
        ground_truth = np.concatenate([context, future_vals], axis=1)
    else:
        assert np.all(ground_truth == np.concatenate([context, future_vals], axis=1)), (
            "Ground truth mismatch!"
        )

    median_predictions = data["median_predictions"][context_window_idx]
    all_predictions = data["predictions"][:, context_window_idx, ...]
    corrected_timesteps = data["corrected_timesteps"]
    edit_prob = data["p"]

    assert edit_prob == p, (
        f"edit probability (p) mismatch! Expected: {p}, Got: {edit_prob}"
    )
    assert median_predictions.shape[-1] == future_vals.shape[-1], (
        "Prediction length mismatch!"
    )

    predictions_decoder_edit_dict[p] = {
        "context": context,
        "future_vals": future_vals,
        "median_predictions": median_predictions,
        "predictions": all_predictions,
        "corrected_timesteps": corrected_timesteps,
        "edit_prob": edit_prob,
    }

    all_metrics_p = load_metrics(p, metrics_dir, "metrics")
    if str(chosen_pred_length) not in all_metrics_p:
        warnings.warn(f"No metrics saved for p={p} and length={chosen_pred_length}")
        continue
    system_metrics = all_metrics_p[str(chosen_pred_length)].get(system_name, None)
    if system_metrics is None:
        warnings.warn(
            f"No metrics for system {system_name} found for p={p} and length={chosen_pred_length}"
        )
        continue
    metrics_dict[p] = system_metrics

In [ ]:
all_metrics_p0 = load_metrics(0.0, metrics_dir, "metrics")
all_metrics_p0["64"].keys()

In [ ]:
all_smapes = [
    all_metrics_p0["512"][system_name]["smape"]
    for system_name in all_metrics_p0["64"].keys()
]
print(len(all_smapes))
print(np.mean(all_smapes))

In [ ]:
all_corrected_timesteps = []
for i, p in enumerate(p_values):
    corrected_timesteps = predictions_decoder_edit_dict[p]["corrected_timesteps"]
    all_corrected_timesteps.append(corrected_timesteps)
    print(f"p: {p}")
    print(f"corrected_timesteps (first 10): {corrected_timesteps[:10]}")
    prev_corrected_timesteps = all_corrected_timesteps[i - 1]
    # find number of True values in prev_corrected_timesteps that are not in corrected_timesteps
    res = np.sum(prev_corrected_timesteps) - np.sum(
        prev_corrected_timesteps & corrected_timesteps
    )
    print(
        f"Number of True values in prev_corrected_timesteps that are not in corrected_timesteps: {res}"
    )

### Plot Ground Truth Trajectory

In [ ]:
if ground_truth is not None:
    save_path = os.path.join(
        plot_save_dir,
        "ground_truth",
        f"{system_name}_3D_ground_truth_context-idx-{context_window_idx}_rseed-{run_rseed}.pdf",
    )
    plot_3d_and_univariate(ground_truth, custom_colors=["black"], save_path=None)

In [ ]:
predictions_decoder_edit_dict[0.0]["median_predictions"].shape

In [ ]:
future_vals.shape

### Plot Predictions

In [ ]:
selected_p = 0.0
dim_to_plot = 0
save_path = os.path.join(
    plot_save_dir,
    f"{system_name}_dim-{dim_to_plot}_p-{selected_p}_context-idx-{context_window_idx}_rseed-{run_rseed}.pdf",
)

plot_single_p_prediction(
    pred_data_dict=predictions_decoder_edit_dict,
    selected_p=selected_p,
    dim_to_plot=dim_to_plot,
    plot_median=False,
    # save_path=save_path,
)

In [ ]:
selected_p = 0.0
dim_to_plot = 0
save_path = os.path.join(
    plot_save_dir,
    f"{system_name}_dim-{dim_to_plot}_p-{selected_p}_context-idx-{context_window_idx}_rseed-{run_rseed}.pdf",
)

plot_single_p_prediction(
    pred_data_dict=predictions_decoder_edit_dict,
    selected_p=selected_p,
    dim_to_plot=dim_to_plot,
    plot_median=False,
    # save_path=save_path,
)

### Plot Forecasts and RMSE at different values of p

In [ ]:
cmap = plt.get_cmap("viridis_r")
colors = cmap(np.linspace(0.05, 0.95, len(p_values)))

In [ ]:
plot_forecast_and_rmse(
    metrics_dict,
    predictions_decoder_edit_dict,
    dim_to_plot=0,
    cmap="Blues",
    figsize=(10, 6),
    save_path=None,
)

In [ ]:
selected_p = 0.3

save_path = os.path.join(
    plot_save_dir,
    "combined",
    f"{system_name}_3D_combined_p-{selected_p}_context-idx-{context_window_idx}_rseed-{run_rseed}.pdf",
)

predictions = predictions_decoder_edit_dict[selected_p]["predictions"]
median_predictions = predictions_decoder_edit_dict[selected_p]["median_predictions"]
# assert np.all(np.median(predictions, axis=0) == median_predictions), (
#     "Predictions and median predictions do not match!"
# )

print(f"Context shape: {context.shape}")
print(f"Future vals shape: {future_vals.shape}")
print(f"All Predictions shape: {predictions.shape}")
print(f"Median predictions shape: {median_predictions.shape}")

plot_3d_and_univariate_with_predictions(
    context,
    future_vals,
    predictions,
    save_path=save_path,
    title=rf"{system_name} ($p_{{\mathrm{{edit}}}}={selected_p}$)",
    title_kwargs={"fontsize": 16},
)

In [ ]:
metrics_dir

In [ ]:
metrics_all_runs = {p: load_metrics(p, metrics_dir, "metrics") for p in p_values}
print(metrics_all_runs)

In [ ]:
selected_metric = "smape"

metric_title_name = selected_metric.upper() if selected_metric != "smape" else "sMAPE"

if selected_metric == "spearman":
    metric_title_name = "1 - Spearman"

all_selected_metric_vals = {
    p: {
        prediction_length: [
            system_metrics[selected_metric]
            for system_metrics in metrics_of_prediction_length.values()
        ]
        for prediction_length, metrics_of_prediction_length in all_metrics_of_p.items()
    }
    for p, all_metrics_of_p in metrics_all_runs.items()
}
print(all_selected_metric_vals)

In [ ]:
for p, pred_length_dict in all_selected_metric_vals.items():
    total_nans = 0
    for pred_length, vals in pred_length_dict.items():
        arr = np.array(vals)
        print(arr.shape)
        n_nans = np.isnan(arr).sum()
        print(f"p={p}, pred_length={pred_length}: number of NaNs = {n_nans}")
        total_nans += n_nans
        # # Remove NaNs from the array and update the dict in place
        # arr_no_nans = arr[~np.isnan(arr)]
        # pred_length_dict[pred_length] = arr_no_nans.tolist()
    print(f"p={p}: total number of NaNs = {total_nans}")


In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))

# Prepare sorted p values and colormap
sorted_p_items = sorted(all_selected_metric_vals.items())
p_values_sorted = [p for p, _ in sorted_p_items]
num_p = len(p_values_sorted)
cmap = mpl.colors.ListedColormap(
    plt.get_cmap("cividis_r")(np.linspace(0.1, 0.9, num_p))
)

for idx, (p, pred_length_dict) in enumerate(sorted_p_items):
    pred_lengths = sorted(map(int, pred_length_dict))
    vals = np.array([pred_length_dict[str(pl)] for pl in pred_lengths])
    # If selected_metric is "spearman", plot 1 - spearman instead
    if selected_metric == "spearman":
        vals = 1 - vals
    means = vals.mean(axis=1)
    stds = vals.std(axis=1)
    ns = vals.shape[1]  # number of samples per prediction length
    print(ns)
    std_err = stds / np.sqrt(ns)
    ax.plot(
        pred_lengths,
        means,
        color=cmap(idx),
        alpha=0.8,
        marker="o",
        markersize=5,
        linewidth=2,
    )
    ax.fill_between(
        pred_lengths, means - std_err, means + std_err, color=cmap(idx), alpha=0.1
    )

ax.set_xlabel("Prediction Length", fontweight="bold")
ax.set_title(metric_title_name, fontweight="bold")
ax.set_xticks(pred_lengths)

# Discrete colorbar for p values
bounds = np.arange(-0.5, num_p + 0.5, 1)
norm = mpl.colors.BoundaryNorm(bounds, cmap.N)
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)
# Reverse the colorbar by flipping the colormap, tick labels, and ticks
reversed_ticks = np.arange(num_p - 1, -1, -1)
cbar = fig.colorbar(
    sm, cax=cax, ticks=reversed_ticks, spacing="proportional", shrink=0.5
)
# Set the tick labels in reversed order
cbar.set_ticklabels([str(p) for p in p_values_sorted[::-1]])
# Invert the colorbar axis so the colors match the reversed labels and ticks
cbar.ax.invert_yaxis()
cbar.set_label(r"$p_{\text{edit}}$", fontweight="bold")

plt.tight_layout()
plt.savefig(
    os.path.join(plot_save_dir, f"{selected_metric}_with_p.pdf"),
    bbox_inches="tight",
)
plt.show()


In [ ]:
# Create a figure with two subplots: main plot and legend below
fig, (ax, ax_legend) = plt.subplots(
    2, 1, figsize=(5, 6), gridspec_kw={"height_ratios": [4, 1]}
)

# Prepare sorted p values and colormap
sorted_p_items = sorted(all_selected_metric_vals.items())
p_values_sorted = [p for p, _ in sorted_p_items]
num_p = len(p_values_sorted)
cmap = mpl.colors.ListedColormap(
    plt.get_cmap("cividis_r")(np.linspace(0.1, 0.9, num_p))
)

# Store handles and labels for legend
handles = []
labels = []

for idx, (p, pred_length_dict) in enumerate(sorted_p_items):
    pred_lengths = sorted(map(int, pred_length_dict))
    vals = np.array([pred_length_dict[str(pl)] for pl in pred_lengths])
    # If selected_metric is "spearman", plot 1 - spearman instead
    if selected_metric == "spearman":
        vals = 1 - vals
    means = vals.mean(axis=1)
    stds = vals.std(axis=1)
    ns = vals.shape[1]  # number of samples per prediction length
    print(ns)
    std_err = stds / np.sqrt(ns)
    (line,) = ax.plot(
        pred_lengths,
        means,
        color=cmap(idx),
        alpha=0.8,
        marker="o",
        markersize=5,
        linewidth=2,
        label=str(p),
    )
    ax.fill_between(
        pred_lengths, means - std_err, means + std_err, color=cmap(idx), alpha=0.1
    )
    handles.append(line)
    labels.append(str(p))

ax.set_xlabel("Prediction Length", fontweight="bold")
ax.set_title(metric_title_name, fontweight="bold")
ax.set_xticks(pred_lengths)

# Hide the legend axis frame and ticks
ax_legend.axis("off")

# Add the legend below the plot, with 4 columns
ax_legend.legend(
    handles,
    [rf"$p_{{\text{{edit}}}}={lbl}$" for lbl in labels],
    loc="center",
    ncol=4,
    frameon=True,
    fontsize=8,
    handlelength=2.0,
    columnspacing=1.2,
)

plt.tight_layout()
plt.savefig(
    os.path.join(plot_save_dir, f"{selected_metric}_with_p.pdf"),
    bbox_inches="tight",
)
plt.show()
